# 04 — Практические границы и доверительные интервалы стоимости

Анализ сохранённых результатов без обучения и симуляции.
Границы μ₁(H), μ₂(H) определяются по точечным оценкам, без интервалов положения μ.
Итоговая таблица содержит V и точечные доверительные интервалы V в выбранных узлах.
Для прежних запусков доступен `ANALYSIS_MODE="legacy"`.


In [ ]:
from dataclasses import replace
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from osfbm.boundaries import boundaries_from_scan
from osfbm.core import NodeResult
from osfbm.results import load_run

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
C1, C2, C3 = "#2a78d6", "#eb6834", "#1baf7a"
INK, MUTED = "#0b0b0b", "#8a8a86"

plt.rcParams.update({
    "figure.figsize": (7.5, 4.2), "figure.dpi": 110,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.edgecolor": MUTED, "axes.labelcolor": INK, "axes.titlesize": 11,
    "axes.grid": True, "grid.color": "#e8e8e4", "grid.linewidth": 0.8,
    "lines.linewidth": 1.8, "text.color": INK,
    "xtick.color": MUTED, "ytick.color": MUTED, "font.size": 9,
    "legend.frameon": False,
})
from osfbm.experiment import load_experiment
from scipy.stats import t
ANALYSIS_MODE = "fresh"


In [ ]:
RUN_ID = "fresh_train20k_test20k_v1"
H = 0.3
EPSILON = 0.03
CONFIDENCE = 0.95
ANALYSIS_MODE = "fresh"
RUN_DIR = ROOT / "result_optimal_stopping" / "runs" / RUN_ID


## Анализ сохранённых оценок

`EPSILON` определяет границы, `CONFIDENCE` — только интервалы стоимости.
Неполная сетка и повторные переходы дают пропуски с отдельной диагностикой.
Все найденные границы отображаются независимо от ширины интервалов V.


In [ ]:
if ANALYSIS_MODE == "fresh":
    if not np.isfinite(EPSILON) or EPSILON <= 0:
        raise ValueError("EPSILON должен быть конечным и положительным")
    if not np.isfinite(CONFIDENCE) or not 0 < CONFIDENCE < 1:
        raise ValueError("CONFIDENCE должен лежать между 0 и 1")

    # Только исходные оценки, без расчёта интервалов μ.
    manifest, nodes, _ = load_experiment(RUN_DIR, epsilon=None)
    horizon = manifest["config"]["T"]
    expected_mu = np.sort(np.asarray(manifest["config"]["mu_grid"], dtype=float))
    halfwidth = t.ppf((1 + CONFIDENCE) / 2, nodes["n"] - 1) * nodes["SE"]
    nodes["ci_low"] = nodes["V"] - halfwidth
    nodes["ci_high"] = nodes["V"] + halfwidth

    def point_boundary(group, side):
        inside = (group["V"] if side == 1 else group["excess"]).to_numpy() <= EPSILON
        changes = np.diff(inside.astype(int))
        if np.any(changes > 0 if side == 1 else changes < 0):
            return None, "Повторные переходы"
        indices = np.flatnonzero(inside)
        if not len(indices):
            return None, "Расширить сетку влево" if side == 1 else "Расширить сетку вправо"
        index = int(indices[-1 if side == 1 else 0])
        if side == 1 and index == len(group) - 1:
            return None, "Расширить сетку вправо"
        if side == 2 and index == 0:
            return None, "Расширить сетку влево"
        return group.iloc[index], "Найдена"

    table_rows, diagnostic_rows = [], []
    for h in sorted(manifest["config"]["H_grid"]):
        group = nodes[nodes.H.eq(h)].sort_values("mu")
        complete = (len(group) >= 2 and group.complete.all()
                    and np.array_equal(group.mu.to_numpy(), expected_mu))
        row, diagnostic = {"H": h}, {"H": h}
        for side in (1, 2):
            selected, status = (point_boundary(group, side) if complete
                                else (None, "Неполная сетка / тест не завершён"))
            row[f"mu{side}"] = np.nan if selected is None else selected["mu"]
            row[f"V_mu{side}"] = np.nan if selected is None else selected["V"]
            row[f"V_mu{side}_ci_low"] = np.nan if selected is None else selected["ci_low"]
            row[f"V_mu{side}_ci_high"] = np.nan if selected is None else selected["ci_high"]
            diagnostic[f"mu{side}_status"] = status
        table_rows.append(row)
        diagnostic_rows.append(diagnostic)
    boundary_values = pd.DataFrame(table_rows)
    diagnostics = pd.DataFrame(diagnostic_rows)

    print("Train:", manifest["config"]["M_train"], "Test:", manifest["config"]["M_test"])
    print("Завершено узлов:", int(nodes.complete.sum()), "/", manifest["family_size"])
    print(f"ε={EPSILON:g}; точечные ДИ V: {100 * CONFIDENCE:g}%")
    display(boundary_values.style.format(precision=5, na_rep="—"))
    missing = diagnostics[(diagnostics.mu1_status != "Найдена")
                          | (diagnostics.mu2_status != "Найдена")]
    if not missing.empty:
        print("Причины пропусков:")
        display(missing)

    OUT = RUN_DIR / "analysis" / f"epsilon_{EPSILON:g}" / f"confidence_{CONFIDENCE:g}"
    OUT.mkdir(parents=True, exist_ok=True)
    nodes.to_csv(OUT / "values.csv", index=False)
    boundary_values.to_csv(OUT / "boundary_values.csv", index=False)
    boundary_values[["H", "mu1", "mu2"]].to_csv(OUT / "boundaries.csv", index=False)
    diagnostics.to_csv(OUT / "boundary_status.csv", index=False)

    fig_boundaries, ax = plt.subplots(figsize=(9, 4.8))
    for side, color in ((1, C1), (2, C2)):
        ax.plot(boundary_values.H, boundary_values[f"mu{side}"],
                ".-", color=color, label=f"μ{side}")
    ax.axhline(0, color=MUTED, linewidth=0.8)
    ax.set(xlabel="H", ylabel="μ", title=f"Практические границы, ε={EPSILON:g}")
    ax.legend()
    fig_boundaries.tight_layout()
    fig_boundaries.savefig(OUT / "boundaries.png", dpi=180, bbox_inches="tight")

    view = nodes[nodes.H.eq(H) & nodes.complete].sort_values("mu")
    if view.empty:
        print(f"Нет завершённых узлов для H={H:g}; общий график и таблица сохранены.")
    else:
        fig_values, axes = plt.subplots(1, 2, figsize=(12, 4.5))
        mu = view.mu.to_numpy()
        positions = np.searchsorted(expected_mu, mu)
        splits = np.flatnonzero(np.diff(positions) != 1) + 1
        for side, ax in enumerate(axes, 1):
            shift = np.zeros(len(view)) if side == 1 else mu * horizon
            means = view.V.to_numpy() - shift
            low = view.ci_low.to_numpy() - shift
            high = view.ci_high.to_numpy() - shift
            for part_no, part in enumerate(np.split(np.arange(len(view)), splits)):
                ax.plot(mu[part], means[part], color=C1,
                        label="Оценка" if part_no == 0 else None)
                ax.fill_between(mu[part], low[part], high[part], color=C1, alpha=0.2,
                                label=f"Точечный ДИ {100 * CONFIDENCE:g}%" if part_no == 0 else None)
            ax.axhline(EPSILON, color=INK, linestyle="--", label=f"ε={EPSILON:g}")
            ax.axhline(0, color=MUTED, linewidth=0.8)
            selected = boundary_values.loc[boundary_values.H.eq(H), f"mu{side}"]
            if len(selected) and pd.notna(selected.iloc[0]):
                ax.axvline(selected.iloc[0], color=C2, linestyle=":", label=f"μ{side}")
            ax.set(xlabel="μ", ylabel="V" if side == 1 else "V − μT",
                   title="Стоимость стратегии" if side == 1 else "Превышение над удержанием")
            ax.legend()
        fig_values.suptitle(f"H={H:g}; ε={EPSILON:g}")
        fig_values.tight_layout()
        fig_values.savefig(OUT / f"values_H_{H:g}.png", dpi=160, bbox_inches="tight")
    plt.show()
    print("Таблицы и графики:", OUT)
elif ANALYSIS_MODE != "legacy":
    raise ValueError("ANALYSIS_MODE должен быть fresh или legacy")


## Как читать результаты

μ₁ — последний узел области V ≤ ε; μ₂ — первый узел области V − μT ≤ ε.
Используются узлы сетки без интерполяции. Повторные переходы дают пропуск с диагностикой.
Уровень доверия на выбор границ не влияет.

Для наград обученной стратегии Y сохранены среднее V и стандартная ошибка SE.
Точечный интервал стоимости: V ± t(n−1, (1+CONFIDENCE)/2) · SE.
В таблице показана **сама стоимость V** при каждой границе, включая μ₂,
а не превышение V − μT. На втором графике интервал сдвинут на известное μT.

Интервалы приближённые и относятся к фиксированной обученной стратегии.
Они не учитывают обучение и дискретизацию, не дают совместного покрытия таблицы
и не корректируют выбор граничного узла по тем же тестовым данным.
Интервалы положения μ не рассчитываются.


In [ ]:
if ANALYSIS_MODE == "legacy":
    # RUN_ID=None позволяет выбрать последний сохранённый запуск.
    if RUN_ID is None:
        runs = sorted((ROOT / "result_optimal_stopping" / "runs").glob("*/config.json"))
        if not runs:
            raise FileNotFoundError("Нет сохранённых запусков. Сначала выполните 03-grid.ipynb")
        RUN_ID = runs[-1].parent.name
    
    RUN_DIR = ROOT / "result_optimal_stopping" / "runs" / RUN_ID
    cfg, nodes = load_run(RUN_DIR)
    view = nodes[nodes["H"].eq(H)].sort_values("mu").copy()
    if view.empty:
        raise ValueError(f"Нет данных для H={H}. Доступные H: {sorted(nodes['H'].unique())}")
    print(f"Запуск: {RUN_ID}; H={H:g}; ε={EPSILON:g}; T={cfg.T}")
    print(f"Сохранено {len(view)} / {len(cfg.mu_grid)} узлов μ для выбранного H")


## Прежний анализ (`ANALYSIS_MODE="legacy"`)

Эти ячейки выполняются только в режиме legacy. Ниже сохранены прежние диагностики.

## Практические границы и значения по μ

Отрицательная разность границ означает перекрытие областей близости к двум
опорным стратегиям. Нелокализованные границы остаются пустыми.

In [ ]:
if ANALYSIS_MODE == "legacy":
    records = [NodeResult(**row) for row in view.to_dict("records")]
    boundary = boundaries_from_scan(records, cfg, epsilon=EPSILON)
    if len(view) < len(cfg.mu_grid):
        boundary = replace(boundary, note=(
            f"Неполная сетка: {len(view)}/{len(cfg.mu_grid)} узлов; " + boundary.note
        ))
    summary = pd.DataFrame([boundary.as_row()])
    display(summary)
    display(view[["mu", "V", "SE", "p0", "p1", "E_tau"]])
    
    OUT = RUN_DIR / "analysis" / f"H_{float(H)}" / f"epsilon_{float(EPSILON)}"
    OUT.mkdir(parents=True, exist_ok=True)
    summary.to_csv(OUT / "boundaries.csv", index=False)


## Стоимость и превышение опорных значений (legacy)

Полоса показывает точечный доверительный интервал стоимости.
Вертикальные линии — найденные ε-границы без интервалов по μ.


In [ ]:
if ANALYSIS_MODE == "legacy":
    mu = view["mu"].to_numpy()
    value = view["V"].to_numpy()
    se = view["SE"].to_numpy()
    halfwidth = t.ppf((1 + CONFIDENCE) / 2, cfg.M_test - 1) * se
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    axes[0].plot(mu, value, color=C1, label="V̂")
    axes[0].fill_between(mu, value - halfwidth, value + halfwidth, color=C1, alpha=0.15, label=f"Точечный ДИ {100 * CONFIDENCE:g}%")
    axes[0].plot(mu, np.maximum(0, mu * cfg.T), "--", color=MUTED, label="max(0, μT)")
    axes[0].set_title("Оценка стоимости")
    axes[0].set_ylabel("V̂")
    axes[1].plot(mu, value, color=C1, label="V̂ − 0")
    axes[2].plot(mu, value - mu * cfg.T, color=C1, label="V̂ − μT")
    for ax, title in zip(axes[1:], ["Превышение над выходом в нуле", "Превышение над удержанием"]):
        ax.axhline(EPSILON, color="black", linestyle="--", label=f"ε={EPSILON:g}")
        ax.axhline(0, color=MUTED, linewidth=0.8)
        ax.set_title(title)
        ax.set_ylabel("Превышение стоимости")
    for position, bracket, label, color, panel in [
        (boundary.mu1_epsilon, boundary.mu1_bracket, "μ₁ᵋ", C2, 1),
        (boundary.mu2_epsilon, boundary.mu2_bracket, "μ₂ᵋ", C3, 2),
    ]:
        if position is not None:
            for ax in (axes[0], axes[panel]):
                ax.axvline(position, color=color, linestyle=":", label=label)
    for ax in axes:
        ax.set_xlabel("μ")
        ax.legend(fontsize=8)
    fig.suptitle(f"H={H:g}, ε={EPSILON:g}")
    fig.tight_layout()
    fig.savefig(OUT / "values.png", dpi=160, bbox_inches="tight")


## Диагностики остановки

Доли остановок, среднее время и численный наклон относятся к той же стратегии.
Близость стоимости к опорной не означает одинаковый момент остановки всех
траекторий; точного совпадения наклона со средним временем здесь не требуется.

In [ ]:
if ANALYSIS_MODE == "legacy":
    fig, ax = plt.subplots()
    ax.plot(mu, view["p0"], label="p0 — выход в нуле")
    ax.plot(mu, view["p1"], label="p1 — удержание до конца")
    ax.plot(mu, view["E_tau"] / cfg.T, label="Среднее время / T")
    if len(view) >= 2:
        ax.plot(mu, np.gradient(value, mu) / cfg.T, "--", label="Численный наклон V̂ / T")
    ax.set(xlabel="μ", ylabel="Доля / нормированный наклон", title=f"Диагностики H={H:g}, ε={EPSILON:g}")
    ax.legend()
    fig.tight_layout()
    fig.savefig(OUT / "diagnostics.png", dpi=160, bbox_inches="tight")
    print(f"Анализ сохранён в {OUT}")


## Границы по всем H — boundaries.png

Этот блок использует все сохранённые H выбранного `RUN_ID` и один заданный
`EPSILON`. Параметр `H` из основного анализа здесь не ограничивает выборку.
Повторного обучения нет. Можно выполнить блок после ячейки загрузки данных.

Слева — практические ε-границы, справа — их разность. Отрицательная разность
означает перекрытие областей близости. Пропуски и нелокализованные границы
отображаются разрывами; подробности приведены в таблице.

Файлы сохраняются в `analysis/epsilon_<EPSILON>/` внутри каталога запуска.

In [ ]:
if ANALYSIS_MODE == "legacy":
    grid_rows = []
    for h in sorted(cfg.H_grid):
        group = nodes[nodes["H"].eq(h)].sort_values("mu")
        if group.empty:
            grid_rows.append({
                "H": h, "epsilon": EPSILON,
                "mu1_epsilon": np.nan, "mu2_epsilon": np.nan,
                "boundary_difference": np.nan,
                "note": "Нет сохранённых узлов для этого H",
            })
            continue
        grid_boundary = boundaries_from_scan(
            [NodeResult(**row) for row in group.to_dict("records")],
            cfg, epsilon=EPSILON,
        )
        if len(group) < len(cfg.mu_grid):
            grid_boundary = replace(grid_boundary, note=(
                f"Неполная сетка: {len(group)}/{len(cfg.mu_grid)} узлов; " + grid_boundary.note
            ))
        grid_rows.append(grid_boundary.as_row())
    
    grid_summary = pd.DataFrame(grid_rows)
    display(grid_summary)
    
    fig_boundaries, axes_boundaries = plt.subplots(1, 2, figsize=(11, 4))
    for column, label, color in [
        ("mu1_epsilon", "μ₁ᵋ", C1),
        ("mu2_epsilon", "μ₂ᵋ", C2),
    ]:
        values = pd.to_numeric(grid_summary[column], errors="coerce").to_numpy(dtype=float)
        axes_boundaries[0].plot(grid_summary["H"], values, marker="o", color=color, label=label)
    difference = pd.to_numeric(grid_summary["boundary_difference"], errors="coerce").to_numpy(dtype=float)
    axes_boundaries[1].plot(grid_summary["H"], difference, marker="D", color=C3, label="μ₂ᵋ − μ₁ᵋ")
    for ax in axes_boundaries:
        ax.axhline(0, color=MUTED, linewidth=0.8)
        ax.set_xlabel("H")
        ax.legend()
    axes_boundaries[0].set(ylabel="μ", title="Практические границы")
    axes_boundaries[1].set(ylabel="Разность границ", title="Отрицательное значение: перекрытие")
    fig_boundaries.suptitle(f"ε={EPSILON:g}, T={cfg.T:g}")
    fig_boundaries.tight_layout()
    
    GRID_OUT = RUN_DIR / "analysis" / f"epsilon_{float(EPSILON)}"
    GRID_OUT.mkdir(parents=True, exist_ok=True)
    grid_summary.to_csv(GRID_OUT / "boundaries.csv", index=False)
    fig_boundaries.savefig(GRID_OUT / "boundaries.png", dpi=200, bbox_inches="tight")
    print(f"График: {GRID_OUT / 'boundaries.png'}")
